# LQG Fragility

LQG fragility: separation principle fails under model uncertainty.

Nominal plant: lightly damped 3rd-order system.
LQG controller designed for nominal A.
Perturb A → A + δA and show closed-loop instability.

Figure: two panels side by side.
  Left:  nominal (stable, good tracking)
  Right: perturbed A (unstable / divergent)

**Figures:** lqg_fragility.svg

In [ ]:
%matplotlib inline

In [ ]:
import numpy as np
from scipy.linalg import solve_continuous_are
from scipy.integrate import solve_ivp
import matplotlib, matplotlib.pyplot as plt

matplotlib.rcParams.update({
    "svg.fonttype": "path", "mathtext.fontset": "cm",
    "font.family": "serif", "font.size": 14,
    "axes.labelsize": 16, "axes.titlesize": 16,
    "legend.fontsize": 11, "xtick.labelsize": 13, "ytick.labelsize": 13,
    "lines.linewidth": 2.2, "axes.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
})
BLUE = "#0072BD"; ORANGE = "#D95319"; GREEN = "#77AC30"

# ── Nominal plant ─────────────────────────────────────────────
# 3rd-order with a lightly damped resonance + integrator-like mode
A = np.array([[ 0,    1,    0  ],
              [ 0,    0,    1  ],
              [-10,  -12,  -2  ]])
B = np.array([[0], [0], [1]])
C = np.array([[1, 0, 0]])
n = 3

eigs_A = np.linalg.eigvals(A)
print(f"Plant eigenvalues: {np.sort_complex(eigs_A).round(3)}")

# ── LQR gain ──────────────────────────────────────────────────
Q_lqr = np.diag([100, 10, 1])
R_lqr = np.array([[0.1]])

P = solve_continuous_are(A, B, Q_lqr, R_lqr)
K = np.linalg.solve(R_lqr, B.T @ P)
eigs_cl = np.linalg.eigvals(A - B @ K)
print(f"LQR closed-loop poles: {np.sort_complex(eigs_cl).round(3)}")

# ── Kalman gain ───────────────────────────────────────────────
W = np.diag([0.1, 0.1, 1.0])   # process noise
V = np.array([[0.5]])           # measurement noise

Pf = solve_continuous_are(A.T, C.T, W, V)
L = (Pf @ C.T @ np.linalg.inv(V))
eigs_obs = np.linalg.eigvals(A - L @ C)
print(f"Kalman observer poles: {np.sort_complex(eigs_obs).round(3)}")

# ── Perturbation: search for destabilising δA ─────────────────
# Perturb the (2,0) entry (affects coupling between modes)
def closed_loop_eigs(delta):
    dA = np.zeros((3, 3))
    dA[2, 0] = delta
    A_true = A + dA
    # Augmented: [A_true-BK, BK; dA, A-LC]
    M = np.block([[A_true - B @ K, B @ K],
                  [dA, A - L @ C]])
    return np.linalg.eigvals(M)

# Sweep to find instability threshold
print("\nSweeping perturbation δA[2,0]:")
for d in [0, 5, 10, 15, 20, 25, 30]:
    eigs = closed_loop_eigs(d)
    max_re = eigs.real.max()
    print(f"  δ = {d:3d}: max Re(λ) = {max_re:+.4f}"
          f"  {'UNSTABLE' if max_re > 0 else 'stable'}")

# Pick a destabilising value
delta_unstable = 25.0
dA = np.zeros((3, 3)); dA[2, 0] = delta_unstable

# Verify
eigs_nom = closed_loop_eigs(0)
eigs_pert = closed_loop_eigs(delta_unstable)
print(f"\nNominal max Re: {eigs_nom.real.max():.4f}")
print(f"Perturbed max Re: {eigs_pert.real.max():.4f}")

# ── Simulate ──────────────────────────────────────────────────
T = 12.0
dt = 0.001
tg = np.arange(0, T, dt)
N = len(tg)
rng = np.random.default_rng(42)

def simulate(A_true, T, tg):
    """Simulate LQG closed-loop with noise."""
    x = np.array([1.0, 0.0, 0.0])   # nonzero IC
    xhat = np.zeros(n)               # observer starts at zero
    ys = np.zeros(N)
    us = np.zeros(N)

    for i in range(N):
        y = (C @ x)[0] + np.sqrt(V[0,0]) * rng.standard_normal()
        u = -(K @ xhat)[0]

        ys[i] = (C @ x)[0]
        us[i] = u

        # Process noise
        w = np.array([np.sqrt(W[0,0]), np.sqrt(W[1,1]),
                       np.sqrt(W[2,2])]) * rng.standard_normal(3)

        # State update (Euler)
        x = x + (A_true @ x + B.flatten() * u + w) * dt
        # Observer update
        innov = y - (C @ xhat)[0]
        xhat = xhat + (A @ xhat + B.flatten() * u + L.flatten() * innov) * dt

    return ys, us

print("Simulating nominal ...", flush=True)
rng = np.random.default_rng(42)
y_nom, u_nom = simulate(A, T, tg)

print("Simulating perturbed ...", flush=True)
rng = np.random.default_rng(42)  # same noise realisation
y_pert, u_pert = simulate(A + dA, T, tg)

# ── Figure ────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), sharey=False)
fig.subplots_adjust(left=0.07, right=0.97, top=0.86, bottom=0.13,
                    wspace=0.25)
fig.suptitle(
    "LQG fragility: separation principle fails under model uncertainty",
    fontsize=14, fontweight="bold")

# Left: nominal
ax1.plot(tg, y_nom, color=BLUE, lw=1.5, alpha=0.9, label=r"$y(t)$")
ax1.axhline(0, color="0.5", lw=0.5)
ax1.set_xlabel("Time $t$")
ax1.set_ylabel(r"Output $y(t)$")
ax1.set_title("Nominal plant ($\\delta A = 0$): stable", fontsize=13)
ax1.set_xlim(0, T)
ax1.grid(True, alpha=0.15)
ax1.legend(loc="upper right", framealpha=0.95, edgecolor="0.75")

# Right: perturbed
ax2.plot(tg, y_pert, color=ORANGE, lw=1.5, alpha=0.9, label=r"$y(t)$")
ax2.axhline(0, color="0.5", lw=0.5)
ax2.set_xlabel("Time $t$")
ax2.set_ylabel(r"Output $y(t)$")
ax2.set_title(
    rf"Perturbed plant ($\delta A_{{3,1}} = {delta_unstable:.0f}$): unstable",
    fontsize=13)
ax2.set_xlim(0, T)
ax2.grid(True, alpha=0.15)
ax2.legend(loc="upper left", framealpha=0.95, edgecolor="0.75")

fig.savefig("lqg_fragility.svg", bbox_inches="tight")
fig.savefig("lqg_fragility.png", dpi=200, bbox_inches="tight")
plt.close()
print("Saved lqg_fragility.svg")